# Statistics for Decision Making — Assignment Solutions
**Dataset:** property.csv (Australian Real Estate Prices)  
**Tool:** Python (pandas, scipy, statsmodels, pingouin)


## Setup — Import Libraries & Load Data

In [12]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_1samp, ttest_ind, binom, f_oneway
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('property.csv')

# Parse Date column
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample:")
df.head(3)


Shape: (13580, 23)

Columns: ['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG', 'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude', 'Longtitude', 'Regionname', 'Propertycount', 'Month', 'Year']

Sample:


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Month,Year
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067.0,...,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0,12,2016
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067.0,...,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0,2,2016
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067.0,...,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0,3,2017


---
## Q1 — One-Sample t-Test: Altona Property Prices
**Claim:** Typical property in Altona sells for \$800,000  
**Test:** One-tailed (right) t-test — has price *increased*?  
**H₀:** μ = 800,000  |  **H₁:** μ > 800,000  |  **α = 0.05**


In [13]:
altona = df[df['Suburb'] == 'Altona']['Price'].dropna()

sample_mean = altona.mean()
sample_std  = altona.std()
n           = len(altona)

# One-sample t-test (two-tailed first, then derive one-tailed)
t_stat, p_two = ttest_1samp(altona, popmean=800000)
p_one = p_two / 2  # right-tailed since t_stat > 0

print(f"Sample Size  : {n}")
print(f"Sample Mean  : ${sample_mean:,.2f}")
print(f"Sample Std   : ${sample_std:,.2f}")
print(f"t-statistic  : {t_stat:.4f}")
print(f"p-value (1-tail, right): {p_one:.4f}")
print()
if p_one < 0.05 and t_stat > 0:
    print("✅ REJECT H₀: There is sufficient evidence at 5% significance that")
    print("   the typical Altona property price has INCREASED above $800,000.")
else:
    print("❌ FAIL TO REJECT H₀: No sufficient evidence that price has increased.")


Sample Size  : 74
Sample Mean  : $834,830.41
Sample Std   : $291,546.05
t-statistic  : 1.0277
p-value (1-tail, right): 0.1537

❌ FAIL TO REJECT H₀: No sufficient evidence that price has increased.


---
## Q2 — Independent Two-Sample t-Test: Summer vs Winter Prices (2016)
**Winter:** Oct–Mar (months 10,11,12,1,2,3)  |  **Summer:** Apr–Sep  
**H₀:** μ_summer = μ_winter  |  **H₁:** μ_summer ≠ μ_winter  |  **α = 0.05**


In [14]:
data_2016 = df[df['Year'] == 2016].copy()

winter_months = [10, 11, 12, 1, 2, 3]
summer_months = [4, 5, 6, 7, 8, 9]

winter = data_2016[data_2016['Month'].isin(winter_months)]['Price'].dropna()
summer = data_2016[data_2016['Month'].isin(summer_months)]['Price'].dropna()

t_stat, p_val = ttest_ind(summer, winter, equal_var=False)  # Welch's t-test

print(f"Summer — n={len(summer)}, mean=${summer.mean():,.2f}, std=${summer.std():,.2f}")
print(f"Winter — n={len(winter)}, mean=${winter.mean():,.2f}, std=${winter.std():,.2f}")
print(f"\nt-statistic : {t_stat:.4f}")
print(f"p-value     : {p_val:.4f}")
print()
if p_val < 0.05:
    print("✅ REJECT H₀: There IS a statistically significant difference in property")
    print("   prices between summer and winter months in 2016.")
else:
    print("❌ FAIL TO REJECT H₀: No statistically significant difference in prices")
    print("   between summer and winter months in 2016.")


Summer — n=4036, mean=$1,048,054.73, std=$621,493.44
Winter — n=2300, mean=$1,116,647.59, std=$695,498.28

t-statistic : -3.9211
p-value     : 0.0001

✅ REJECT H₀: There IS a statistically significant difference in property
   prices between summer and winter months in 2016.


---
## Q3 — Binomial Probability: No Car Parking in Abbotsford
**Question:** Out of 10 properties sold in Abbotsford, P(exactly 3 have NO car parking)?  
**Approach:** Binomial distribution — B(n=10, k=3, p=P(no car))


In [4]:
abbotsford = df[df['Suburb'] == 'Abbotsford']['Car'].dropna()

p_no_car = (abbotsford == 0).sum() / len(abbotsford)
print(f"Total Abbotsford records (with Car data): {len(abbotsford)}")
print(f"Properties with NO car parking: {(abbotsford == 0).sum()}")
print(f"P(no car parking) = {p_no_car:.4f}")

# P(X = 3) from Binomial(n=10, p=p_no_car)
prob = binom.pmf(k=3, n=10, p=p_no_car)
print(f"\nP(exactly 3 out of 10 have no car parking) = {prob:.3f}")


Total Abbotsford records (with Car data): 56
Properties with NO car parking: 15
P(no car parking) = 0.2679

P(exactly 3 out of 10 have no car parking) = 0.260


---
## Q4 — Empirical Probability: Property with 3 Rooms in Abbotsford


In [5]:
ab_rooms = df[df['Suburb'] == 'Abbotsford']['Rooms'].dropna()

p_3_rooms = (ab_rooms == 3).sum() / len(ab_rooms)
print(f"Total Abbotsford properties: {len(ab_rooms)}")
print(f"Properties with 3 rooms    : {(ab_rooms == 3).sum()}")
print(f"\nP(3 rooms) = {p_3_rooms:.3f}")


Total Abbotsford properties: 56
Properties with 3 rooms    : 20

P(3 rooms) = 0.357


---
## Q5 — Empirical Probability: Property with 2 Bathrooms in Abbotsford


In [6]:
ab_bath = df[df['Suburb'] == 'Abbotsford']['Bathroom'].dropna()

p_2_bath = (ab_bath == 2).sum() / len(ab_bath)
print(f"Total Abbotsford properties (with bathroom data): {len(ab_bath)}")
print(f"Properties with 2 bathrooms : {(ab_bath == 2).sum()}")
print(f"\nP(2 bathrooms) = {p_2_bath:.3f}")


Total Abbotsford properties (with bathroom data): 56
Properties with 2 bathrooms : 19

P(2 bathrooms) = 0.339


---
## Q6 — One-Sample t-Test: Richmond Average Price
**Claim:** Average property price in Richmond = \$1,000,000  
**H₀:** μ = 1,000,000  |  **H₁:** μ ≠ 1,000,000  |  **α = 0.05**


In [7]:
richmond = df[df['Suburb'] == 'Richmond']['Price'].dropna()

t_stat, p_val = ttest_1samp(richmond, popmean=1_000_000)

print("=" * 50)
print("ONE-SAMPLE t-TEST — Richmond Property Prices")
print("=" * 50)
print(f"H₀: μ = $1,000,000")
print(f"H₁: μ ≠ $1,000,000  (two-tailed)")
print(f"Significance level (α): 0.05")
print()
print(f"Sample Size  : {len(richmond)}")
print(f"Sample Mean  : ${richmond.mean():,.2f}")
print(f"Sample Std   : ${richmond.std():,.2f}")
print(f"t-statistic  : {t_stat:.4f}")
print(f"p-value      : {p_val:.4f}")
print()
if p_val < 0.05:
    print("✅ REJECT H₀")
    print("   Business Conclusion: The actual average Richmond property price is")
    print("   SIGNIFICANTLY DIFFERENT from the claimed $1,000,000.")
    direction = "above" if richmond.mean() > 1_000_000 else "below"
    print(f"   The data suggests prices are {direction} the claimed benchmark.")
else:
    print("❌ FAIL TO REJECT H₀")
    print("   Business Conclusion: The data does not provide sufficient evidence")
    print("   that the average price differs from $1,000,000.")


ONE-SAMPLE t-TEST — Richmond Property Prices
H₀: μ = $1,000,000
H₁: μ ≠ $1,000,000  (two-tailed)
Significance level (α): 0.05

Sample Size  : 260
Sample Mean  : $1,083,564.42
Sample Std   : $522,353.52
t-statistic  : 2.5795
p-value      : 0.0104

✅ REJECT H₀
   Business Conclusion: The actual average Richmond property price is
   SIGNIFICANTLY DIFFERENT from the claimed $1,000,000.
   The data suggests prices are above the claimed benchmark.


---
## Q7 — Two-Sample t-Test: Car Parking Impact on Price
**H₀:** μ_with_car = μ_without_car  |  **H₁:** μ_with_car > μ_without_car  |  **α = 0.05**  
**Test Choice:** Welch's independent t-test (unequal sample sizes & variances expected)


In [8]:
car_data = df[['Car', 'Price']].dropna()

with_car    = car_data[car_data['Car'] > 0]['Price']
without_car = car_data[car_data['Car'] == 0]['Price']

# One-tailed: does with_car > without_car?
t_stat, p_two = ttest_ind(with_car, without_car, equal_var=False)
p_one = p_two / 2  # right-tailed since we expect with_car > without_car

print("=" * 55)
print("TWO-SAMPLE t-TEST — Car Parking vs No Car Parking")
print("=" * 55)
print(f"With Car    — n={len(with_car):,}, mean=${with_car.mean():,.2f}")
print(f"Without Car — n={len(without_car):,}, mean=${without_car.mean():,.2f}")
print()
print(f"Test chosen  : Welch's two-sample t-test (unequal variances)")
print(f"t-statistic  : {t_stat:.4f}")
print(f"p-value (1-tail): {p_one:.6f}")
print()
if p_one < 0.05 and t_stat > 0:
    print("✅ REJECT H₀: Properties WITH car parking sell at a SIGNIFICANTLY")
    print("   HIGHER average price than those without.")
    print()
    print("   Business Implication for Developers:")
    print("   → Car parking is a value-adding feature. Including parking in")
    print("     new developments is likely to justify higher asking prices.")
    print("     Developers should factor parking into project planning.")
else:
    print("❌ FAIL TO REJECT H₀")


TWO-SAMPLE t-TEST — Car Parking vs No Car Parking
With Car    — n=12,492, mean=$1,074,443.92
Without Car — n=1,026, mean=$1,079,088.01

Test chosen  : Welch's two-sample t-test (unequal variances)
t-statistic  : -0.2722
p-value (1-tail): 0.392737

❌ FAIL TO REJECT H₀


---
## Q8 — Two-Way ANOVA: Suburb × Property Type Impact on Price
**Factors:** Suburb (A), Property Type (B), Interaction (A×B)  
**H₀ for each:** The factor has no effect on price  
*(Using top 5 suburbs by count for computational tractability)*


In [9]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# Use top 5 suburbs for manageable ANOVA
top5_suburbs = df['Suburb'].value_counts().head(5).index.tolist()
anova_df = df[df['Suburb'].isin(top5_suburbs)][['Suburb', 'Type', 'Price']].dropna()

print(f"Records used: {len(anova_df)}")
print(f"Suburbs     : {top5_suburbs}")
print(f"Types       : {anova_df['Type'].unique().tolist()}")
print()

# Fit OLS model with interaction
model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)', data=anova_df).fit()
anova_table = anova_lm(model, typ=2)

print("=" * 60)
print("TWO-WAY ANOVA RESULTS")
print("=" * 60)
print(anova_table.round(4))
print()

# Interpret
alpha = 0.05
for factor in anova_table.index[:-1]:  # skip Residual
    p = anova_table.loc[factor, 'PR(>F)']
    sig = "✅ SIGNIFICANT" if p < alpha else "❌ NOT significant"
    print(f"{factor.strip():40s} p={p:.4f}  →  {sig}")

print()
print("Interpretation:")
print("→ If Suburb is significant: location drives price differences.")
print("→ If Type is significant: property type (house/unit/townhouse) affects price.")
print("→ If Interaction is significant: the type effect differs across suburbs.")


Records used: 1329
Suburbs     : ['Reservoir', 'Richmond', 'Bentleigh East', 'Preston', 'Brunswick']
Types       : ['h', 't', 'u']

TWO-WAY ANOVA RESULTS
                         sum_sq      df         F  PR(>F)
C(Suburb)          4.160952e+13     4.0  139.1428     0.0
C(Type)            6.480908e+13     2.0  433.4449     0.0
C(Suburb):C(Type)  6.129412e+12     8.0   10.2484     0.0
Residual           9.823523e+13  1314.0       NaN     NaN

C(Suburb)                                p=0.0000  →  ✅ SIGNIFICANT
C(Type)                                  p=0.0000  →  ✅ SIGNIFICANT
C(Suburb):C(Type)                        p=0.0000  →  ✅ SIGNIFICANT

Interpretation:
→ If Suburb is significant: location drives price differences.
→ If Type is significant: property type (house/unit/townhouse) affects price.
→ If Interaction is significant: the type effect differs across suburbs.


---
## Q9 — p-Value Interpretation (Conceptual)
**Scenario:** A hypothesis test comparing prices across two suburbs gives p = 0.032


In [10]:
p_value = 0.032
alpha   = 0.05

print("=" * 55)
print("p-VALUE INTERPRETATION")
print("=" * 55)
print(f"Given p-value : {p_value}")
print(f"Significance α: {alpha}")
print()
print("1. What does p = 0.032 mean?")
print("   → If the null hypothesis (no price difference) were true,")
print("     there is only a 3.2% probability of observing a test")
print("     statistic as extreme as the one obtained. This is a")
print("     relatively rare outcome under H₀.")
print()
print("2. Should H₀ be rejected at α = 0.05?")
decision = "YES — REJECT H₀" if p_value < alpha else "NO — FAIL TO REJECT H₀"
print(f"   → {decision}")
print(f"     Since p ({p_value}) < α ({alpha}), we reject the null hypothesis.")
print()
print("3. Business Stakeholder Interpretation:")
print("   → There IS a statistically significant difference in property")
print("     prices between the two suburbs. A business stakeholder")
print("     should treat these suburbs as DISTINCT pricing markets —")
print("     pricing strategy, investment decisions, and valuation")
print("     models should account for this difference.")


p-VALUE INTERPRETATION
Given p-value : 0.032
Significance α: 0.05

1. What does p = 0.032 mean?
   → If the null hypothesis (no price difference) were true,
     there is only a 3.2% probability of observing a test
     statistic as extreme as the one obtained. This is a
     relatively rare outcome under H₀.

2. Should H₀ be rejected at α = 0.05?
   → YES — REJECT H₀
     Since p (0.032) < α (0.05), we reject the null hypothesis.

3. Business Stakeholder Interpretation:
   → There IS a statistically significant difference in property
     prices between the two suburbs. A business stakeholder
     should treat these suburbs as DISTINCT pricing markets —
     pricing strategy, investment decisions, and valuation
     models should account for this difference.


---
## Q10 — Hypothesis Validation: 2+ Bathrooms Command Premium Price
**Claim:** Properties with MORE than 2 bathrooms are priced higher  
**Test:** Welch's one-tailed two-sample t-test  
**H₀:** μ_(>2 bath) ≤ μ_(≤2 bath)  |  **H₁:** μ_(>2 bath) > μ_(≤2 bath)  |  **α = 0.05**


In [11]:
bath_data = df[['Bathroom', 'Price']].dropna()

premium  = bath_data[bath_data['Bathroom'] > 2]['Price']
standard = bath_data[bath_data['Bathroom'] <= 2]['Price']

t_stat, p_two = ttest_ind(premium, standard, equal_var=False)
p_one = p_two / 2  # right-tailed

print("=" * 58)
print("HYPOTHESIS TEST — Bathroom Count & Price Premium")
print("=" * 58)
print(f"H₀: μ(>2 bathrooms) ≤ μ(≤2 bathrooms)")
print(f"H₁: μ(>2 bathrooms) > μ(≤2 bathrooms)  [one-tailed]")
print(f"α  = 0.05")
print()
print(f"More than 2 baths — n={len(premium):,}, mean=${premium.mean():,.2f}")
print(f"Up to 2 baths     — n={len(standard):,}, mean=${standard.mean():,.2f}")
print()
print(f"Test Used    : Welch's two-sample t-test (one-tailed)")
print(f"t-statistic  : {t_stat:.4f}")
print(f"p-value (1-tail): {p_one:.6f}")
print()
if p_one < 0.05 and t_stat > 0:
    print("✅ REJECT H₀")
    print("   Conclusion: Properties with more than 2 bathrooms command a")
    print("   STATISTICALLY SIGNIFICANT price premium.")
    print()
    print("   Recommendation to Policymakers:")
    print("   → Bathroom count is a significant price driver. Housing affordability")
    print("     policies should account for this premium — e.g., stamp duty")
    print("     brackets or first-home buyer schemes could differentiate based")
    print("     on property specification rather than just price alone.")
else:
    print("❌ FAIL TO REJECT H₀: Insufficient evidence of a premium.")


HYPOTHESIS TEST — Bathroom Count & Price Premium
H₀: μ(>2 bathrooms) ≤ μ(≤2 bathrooms)
H₁: μ(>2 bathrooms) > μ(≤2 bathrooms)  [one-tailed]
α  = 0.05

More than 2 baths — n=1,060, mean=$1,882,824.20
Up to 2 baths     — n=12,520, mean=$1,007,347.94

Test Used    : Welch's two-sample t-test (one-tailed)
t-statistic  : 28.8002
p-value (1-tail): 0.000000

✅ REJECT H₀
   Conclusion: Properties with more than 2 bathrooms command a
   STATISTICALLY SIGNIFICANT price premium.

   Recommendation to Policymakers:
   → Bathroom count is a significant price driver. Housing affordability
     policies should account for this premium — e.g., stamp duty
     brackets or first-home buyer schemes could differentiate based
     on property specification rather than just price alone.
